In [ ]:
import scanpy as sc
import pandas as pd
import gc
import numpy as np
import os
import math
import anndata as ad
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

def apply_filters(adata, mt_frac=5, min_counts=1000, max_counts=0.99):

    # filtered
    filtered_obs = adata[~((adata.obs['pct_counts_mt'] < mt_frac) &\
                    (adata.obs['tscp_count'] >= min_counts) &\
                    (adata.obs['tscp_count'] < adata.obs['tscp_count'].quantile(max_counts)))].obs

    # retained
    adata = adata[(adata.obs['pct_counts_mt'] < mt_frac) &\
                    (adata.obs['tscp_count'] >= min_counts) &\
                    (adata.obs['tscp_count'] < adata.obs['tscp_count'].quantile(max_counts))]
    adata = adata.copy()

    return adata, filtered_obs

/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
output_dir = '../data/validations/parse'

samples = [
'GBM_RNA16719639',
'GBM_RNA16719636',
'GBM_RNA16719632',
'GBM_RNA16719633',
'GBM_RNA16719635',
'GBM_RNA16719638',
'GBM_RNA16719634',
'GBM_RNA16719637',
'GBM_RNA16567069',
'GBM_RNA16567070',
'GBM_RNA16567071',
'GBM_RNA16567072',
'GBM_RNA16567073',
'GBM_RNA16567074',
'GBM_RNA16567075',
'GBM_RNA16567076',
]

url = "https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt"
genes = pd.read_csv(url, header=None)[0].tolist()
s_genes  = genes[:43]
g2m_genes = genes[43:]

barcode_sample_dict = {
    'cd44- E25-MOI: 0.125':['A'+str(i) for i in range(1,5)],
    'cd44- E25-MOI: 0.25':['A'+str(i) for i in range(5,9)],
    'cd44- E25-MOI:0.5':['A'+str(i) for i in range(9,13)],
    'cd44- E30-MOI: 0.125':['B'+str(i) for i in range(1,5)],
    'cd44- E30-MOI: 0.25':['B'+str(i) for i in range(5,9)],
    'cd44- E30-MOI:0.5':['B'+str(i) for i in range(9,13)],
    'cd11b- bulk':['C'+str(i) for i in range(1,5)],
    'cd11b- cd44+':['C'+str(i) for i in range(5,9)],
    'cd11b- cd44-':['C'+str(i) for i in range(9,13)],
    'E25 baseline':['D'+str(i) for i in range(1,4)],
    'E30 baseine':['D'+str(i) for i in range(4,7)],
    'cd44- E25':['D'+str(i) for i in range(7,10)],
    'cd44- E30':['D'+str(i) for i in range(10,13)]
}

barcode_sample_dict = {
    item: key
    for key, values in barcode_sample_dict.items()
    for item in values
}

# QC

In [ ]:
ads = []
filtered = []
for sample in samples:
    input_dir = '../../../data/parse_output/{}/all-sample/DGE_filtered'.format(sample)
    sc_path = os.path.join(input_dir, "count_matrix.mtx")
    
    # load mtx
    adata = sc.read_mtx(sc_path)
    
    # var
    adata.var = pd.read_csv(os.path.join(input_dir, "all_genes.csv"))
    adata.var.set_index("gene_name", inplace=True)
    adata.var_names_make_unique()
    
    # obs
    adata.obs = pd.read_csv(os.path.join(input_dir, "cell_metadata.csv"))
    adata.obs.set_index("bc_wells", inplace=True) # barcode column used by Parse

    # sample ID
    adata.obs['sample'] = sample
    
    # QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

    # filters
    adata, filtered_obs = apply_filters(adata)

    # doublets
    sc.pp.scrublet(adata)
    
    gc.collect()
    ads.append(adata)
    filtered.append(filtered_obs)

# merge object and save counts sep
adata = ad.concat(ads, merge = 'same')
adata.layers['counts'] = adata.X.copy()

# Get cell cycle score
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# score cell cycle
sc.tl.score_genes_cell_cycle(
        adata,
        s_genes=s_genes,
        g2m_genes=g2m_genes
    )
adata.var['cell_cycle'] = adata.var_names.isin(genes)

adata.obs['sample_name'] = [barcode_sample_dict[i] for i in adata.obs['bc1_well']]
adata.obs['bc_wells'] = adata.obs.index
adata.obs['cell_id'] = adata.obs['bc_wells']+'_'+adata.obs['sample'].astype(str)+'_'+adata.obs['sample_name'].astype(str)
adata.obs = adata.obs.drop('bc_wells', axis=1)

adata.X = adata.layers['counts']
adata.write_h5ad(os.path.join(output_dir, 'adata_query_QC_batch3.h5ad'))

/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
# filtered assessment

filtered_obs = pd.concat(filtered)
filtered_obs['sample_name'] = [barcode_sample_dict[i] for i in filtered_obs['bc1_well']]
filtered_obs['bc_wells'] = filtered_obs.index
filtered_obs['cell_id'] = filtered_obs['bc_wells']+'_'+filtered_obs['sample'].astype(str)+'_'+filtered_obs['sample_name'].astype(str)

In [ ]:
filtered_obs.groupby(['sample_name']).agg({'cell_id':'count', 
                                        'tscp_count':'mean',
                                        'gene_count':'mean'}).round(1)\
    .rename(columns={'cell_id':'Cell count', 'tscp_count':'mean TSCP (like UMI)'})

,Cell count,mean TSCP (like UMI),gene_count
sample_name,,,
E25 baseline,134,34655.6,6498.9
E30 baseine,932,38172.0,6241.6
cd11b- bulk,288,1830.4,887.8
cd11b- cd44+,103,1306.8,801.5
cd11b- cd44-,154,945.0,734.3
cd44- E25,330,26132.5,5038.3
cd44- E25-MOI: 0.125,249,9062.8,2119.2
cd44- E25-MOI: 0.25,446,14326.1,2974.7
cd44- E25-MOI:0.5,185,17814.6,3719.9


In [ ]:
adata.obs.groupby(['sample_name']).agg({'cell_id':'count', 
                                        'tscp_count':'mean',
                                        'gene_count':'mean'}).round(1)\
    .rename(columns={'cell_id':'Cell count', 'tscp_count':'mean TSCP (like UMI)'})

/tmp/ipykernel_1299267/4107665050.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(['sample_name']).agg({'cell_id':'count',


,Cell count,mean TSCP (like UMI),gene_count
sample_name,,,
E25 baseline,15963,6755.6,2984.2
E30 baseine,12801,6405.3,2679.4
cd11b- bulk,7143,4139.1,2254.0
cd11b- cd44+,5580,4516.1,2389.5
cd11b- cd44-,2115,3078.6,1808.6
cd44- E25,22842,6303.7,2806.2
cd44- E25-MOI:0.5,5535,6198.8,2824.5
cd44- E25-MOI: 0.25,8068,5103.5,2474.6
cd44- E25-MOI: 0.125,10463,5050.8,2509.6
